# Noise generation — building the perturbation gallery

This notebook produces the single-kernel perturbed frames that `noise_removal.ipynb` will try to denoise. Along the way you will:

1. Load a clean CAMUS ultrasound frame from DICOM.
2. Segment the **ultrasound sector** (the fan-shaped region of valid signal) so that every subsequent operation is spatially scoped.
3. Walk through the **deep-learning preprocessing pipeline** (square padding → resize → ImageNet normalisation → sector-only masking) that the DL denoiser in the next notebook expects.
4. Implement five perturbation kernels (`gaussian_blur`, `contrast`, `speckle`, `gaussian_noise`, `salt_pepper`) and save one PNG per kernel to `./lab-0-data/perturbed_images/`.

All perturbations are photometric and act **inside the sector only**; the dark background is preserved so that boundary artefacts do not leak into downstream analysis.

## Environment setup

In [ ]:
!pip install pydicom

In [ ]:
import logging
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from pydicom import dcmread
from scipy.ndimage import binary_closing, gaussian_filter
from scipy.spatial import ConvexHull, QhullError
from skimage.draw import polygon
from skimage.metrics import structural_similarity
import os

logger = logging.getLogger(__name__)

In [ ]:
# Reproducibility seed used by every RNG in this notebook.
SEED = 42

# Sector-mask parameters.
_THRESHOLD = 0.01  # foreground/background split on the normalised frame
_CLOSE_RADIUS = 3  # radius (px) of the closing structuring element

# DL-preprocessing target: 224x224 with ImageNet channel statistics.
_MODEL_INPUT_SIZE = 224
_IMAGENET_MEAN = (0.485, 0.456, 0.406)
_IMAGENET_STD = (0.229, 0.224, 0.225)

# SSIM Gaussian-window size (must be <= sector bbox to be meaningful).
_SSIM_WIN = 7

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the drive containing the images
    drive.mount('/content/drive')

    # List contents (IPython magic command)
    !ls

In [ ]:
if IN_COLAB:
    # Change the working to the current notebook's directory
    %cd /content/drive/MyDrive/
else:
    # Ask the user to fill in the base data folder path
    base_data_folder = input("Please enter the path to the base data folder path: ")
    os.chdir(base_data_folder)

## Ultrasound sector mask

Every ultrasound frame contains a fan-shaped **sector** of valid acoustic signal, surrounded by a black background (device UI, burn-in text, etc.). Restricting all operations — perturbation kernels, normalisation, quality metrics — to the sector avoids boundary artefacts and prevents the background from biasing statistics such as PSNR.

The mask is built with a robust pipeline:
1. Threshold the (normalised) frame at a small value.
2. Apply a morphological **closing** with a disk kernel to bridge small dropouts.
3. Wrap the surviving pixels in their **convex hull** so that dark chamber interiors are still kept inside the sector.

In [ ]:
def _disk_structuring_element(radius: int) -> np.ndarray:
    """Build a 2D boolean disk of the given radius for morphology.

    Args:
        radius: Disk radius in pixels.

    Returns:
        Boolean array of shape ``(2*radius + 1, 2*radius + 1)`` with
        ``True`` inside the disk.
    """
    # Open grid (sparse, memory-cheap) centred at (0, 0).
    y, x = np.ogrid[-radius : radius + 1, -radius : radius + 1]
    # Standard circle equation.
    return (x**2 + y**2) <= radius**2

In [ ]:
def compute_sector_mask(image: np.ndarray) -> np.ndarray:
    """Extract the ultrasound sector via threshold + closing + convex hull.

    The input is min-max normalised internally, so the function is
    robust to whatever dynamic range the DICOM pixel array happens to
    have (uint8, uint16, already-normalised float, ...).

    Args:
        image: 2D float image of arbitrary dynamic range.

    Returns:
        Boolean mask with ``True`` inside the sector.
    """
    if image.ndim != 2:
        raise ValueError(f"Expected 2-D image, got shape {image.shape}")

    # Min-max normalise so `_THRESHOLD` has a consistent meaning regardless
    # of the caller's pixel range.
    img_min = float(image.min())
    img_max = float(image.max())
    if img_max <= img_min:
        logger.warning("compute_sector_mask: flat image; returning empty mask")
        return np.zeros(image.shape, dtype=bool)
    normalised = (image.astype(np.float32) - img_min) / (img_max - img_min)

    # 1. Threshold: everything meaningfully above zero is candidate foreground.
    binary = normalised > _THRESHOLD
    if not binary.any():
        logger.warning("compute_sector_mask: all-zero frame; returning empty mask")
        return np.zeros(image.shape, dtype=bool)

    # 2. Morphological closing: bridge tiny gaps and smooth frayed edges.
    binary = binary_closing(binary, structure=_disk_structuring_element(_CLOSE_RADIUS))

    # 3. Convex hull: rubber-band the outermost pixels so dark interiors
    #    (blood pool, chambers) are still part of the sector.
    rows, cols = np.nonzero(binary)
    points = np.column_stack([rows, cols])
    if len(points) < 3:
        return binary.astype(bool)
    try:
        hull = ConvexHull(points)
    except QhullError:
        # Collinear / degenerate points — fall back to the raw threshold mask.
        return binary.astype(bool)

    hull_vertices = points[hull.vertices]
    rr, cc = polygon(hull_vertices[:, 0], hull_vertices[:, 1], shape=image.shape)
    mask = np.zeros(image.shape, dtype=bool)
    mask[rr, cc] = True
    return mask

## Data loading

The lab data lives in `./lab-0-data/camus/` relative to this notebook. Each `.dcm` file is a single CAMUS frame stored as a Secondary Capture DICOM (grayscale, single-slice). The next cell lists what is available; the one after picks a single frame that will serve as the running example throughout this notebook.

In [ ]:
!ls

In [ ]:
DATA_PATH = Path("./lab-0-data")
INPUT_FOLDER_PATH = DATA_PATH / "camus"

dicom_files = sorted(INPUT_FOLDER_PATH.glob("*.dcm"))
print(f"Found {len(dicom_files)} DICOM file(s) in {INPUT_FOLDER_PATH}")
for f in dicom_files:
    print(f.name)

In [ ]:
selected_file = INPUT_FOLDER_PATH / "0001.dcm"
ds = dcmread(selected_file)

# Quick peek at the DICOM metadata. Skip the raw pixel array — it is
# printed as a huge byte blob and rarely useful at inspection time.
for elem in ds:
    if elem.tag == (0x7FE0, 0x0010):
        continue
    print(f"{elem.tag} {elem.name}: {elem.value}")

## Why the sector mask matters — a synthetic demo

Before applying any real perturbation, we illustrate what the sector mask buys us. We deliberately corrupt the **background** of the frame by painting three white boxes into the corners — a caricature of the burn-in text, callipers and UI overlays that a real ultrasound console can leave outside the acoustic sector.

The rest of Section 5 (padding → resize → normalise → mask) will then show that multiplying by the sector mask cleanly wipes those corner boxes out, so downstream operations only ever see valid acoustic signal.

In [ ]:
# Copy the pixel array so the original DICOM stays untouched, and
# cast to float32 for all downstream numeric work.
image = ds.pixel_array.astype(np.float32).copy()
sector_mask = compute_sector_mask(image)
H, W = image.shape

# Paint three white boxes (~10 % of frame size) into the background
# corners. These are the synthetic "UI overlay" artefacts the sector
# mask will suppress later.
box_h, box_w = H // 10, W // 10
white_val = float(image.max())
image[:box_h, :box_w] = white_val  # top-left
image[:box_h, W - box_w :] = white_val  # top-right
image[H - box_h :, W - box_w :] = white_val  # bottom-right

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(image, cmap="gray")
ax.set_title("Frame with synthetic background artefacts")
ax.axis("off")
plt.show()

## Preprocessing pipeline for the DL denoiser

The DL denoiser you will use in `noise_removal.ipynb` expects 224x224 RGB tensors, ImageNet-normalised, with the background zeroed out. This section builds that pipeline step by step on the corrupted frame from the previous cell.

### 1. Square padding

US frames are rectangular but the network wants square inputs. We pad the shorter side with zeros and also return a boolean **pad mask** marking which pixels come from the original frame — essential later when we want to null out padded regions after channel-wise normalisation.

In [ ]:
def pad_to_square(arr: np.ndarray, *, pad_value: float = 0.0) -> np.ndarray:
    """Zero-pad the shorter side so the output is square.

    Args:
        arr: 2D or 3D array (H, W[, C]).
        pad_value: Fill value for the added rows/columns.

    Returns:
        Square array with the original content centred.
    """
    h, w = arr.shape[:2]
    if h == w:
        return arr
    side = max(h, w)
    top = (side - h) // 2
    bottom = (side - h) - top
    left = (side - w) // 2
    right = (side - w) - left
    pad_width: list[tuple[int, int]] = [(top, bottom), (left, right)]
    if arr.ndim == 3:
        pad_width.append((0, 0))
    return np.pad(arr, pad_width, mode="constant", constant_values=pad_value)


def pad_to_square_with_mask(
    arr: np.ndarray, *, pad_value: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    """Pad to square and return a boolean mask marking original content.

    Args:
        arr: 2D or 3D array (H, W[, C]).
        pad_value: Fill value for the added rows/columns.

    Returns:
        ``(padded_array, pad_mask)`` where ``pad_mask[i, j]`` is
        ``True`` iff ``padded_array[i, j]`` belongs to the original frame.
    """
    h, w = arr.shape[:2]
    if h == w:
        return arr, np.ones((h, w), dtype=bool)
    padded = pad_to_square(arr, pad_value=pad_value)
    top = (padded.shape[0] - h) // 2
    left = (padded.shape[1] - w) // 2
    mask = np.zeros(padded.shape[:2], dtype=bool)
    mask[top : top + h, left : left + w] = True
    return padded, mask

In [ ]:
prev_image_shape = image.shape
prev_sector_mask_shape = sector_mask.shape
padded, pad_mask = pad_to_square_with_mask(image, pad_value=0.0)
padded_sector = pad_to_square(sector_mask.astype(np.uint8), pad_value=0)
print(f"Image        : {prev_image_shape} -> {padded.shape}")
print(f"Sector mask  : {prev_sector_mask_shape} -> {padded_sector.shape}")

### 2. Resize to 224x224

The image is resized with **bilinear** interpolation (smooth) and the masks with **nearest-neighbour** interpolation (keeps values strictly 0/1, no blurred fractional edges).

Look at the axes to see the effect of downscaling: the full images look identical, but a centred 10 % zoom exposes the bilinear smoothing on the resized version.

In [ ]:
def resize_bilinear(arr: np.ndarray, size: int) -> np.ndarray:
    """Resize a 2D float array to ``(size, size)`` with PIL bilinear.

    Used for continuous image data where smooth gradients matter.
    """
    im = Image.fromarray(arr.astype(np.float32), mode="F")
    im = im.resize((size, size), resample=Image.Resampling.BILINEAR)
    return np.array(im, dtype=np.float32, copy=True)


def resize_nearest(arr: np.ndarray, size: int) -> np.ndarray:
    """Resize a 2D uint8 mask to ``(size, size)`` with PIL nearest-neighbour.

    Used for binary masks: preserves sharp boundaries and strict 0/1 values.
    """
    im = Image.fromarray(arr.astype(np.uint8), mode="L")
    im = im.resize((size, size), resample=Image.Resampling.NEAREST)
    return np.array(im, dtype=np.uint8, copy=True)

In [ ]:
prev_image = padded.copy()
padded = resize_bilinear(padded, _MODEL_INPUT_SIZE)

print(f"Shape before resizing: {prev_image.shape}")
print(f"Shape after resizing : {padded.shape}")

# Full images + centred 10 % zoom crops.
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(prev_image, cmap="gray")
axes[0, 0].set_title(f"Padded {prev_image.shape}")
axes[0, 1].imshow(padded, cmap="gray")
axes[0, 1].set_title(f"Resized {padded.shape}")


def _center_crop(arr: np.ndarray, frac: float = 0.10) -> np.ndarray:
    h, w = arr.shape
    r0, r1 = int(h * (0.5 - frac / 2)), int(h * (0.5 + frac / 2))
    c0, c1 = int(w * (0.5 - frac / 2)), int(w * (0.5 + frac / 2))
    return arr[r0:r1, c0:c1]


axes[1, 0].imshow(_center_crop(prev_image), cmap="gray")
axes[1, 0].set_title("Zoom — padded (native pixels)")
axes[1, 1].imshow(_center_crop(padded), cmap="gray")
axes[1, 1].set_title("Zoom — resized (bilinear smoothing)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3. Resize the masks (nearest-neighbour)

In [ ]:
prev_pad_mask_shape = pad_mask.shape
prev_sector_shape = padded_sector.shape
pad_mask_r = resize_nearest(pad_mask.astype(np.uint8), _MODEL_INPUT_SIZE)
sector_r = resize_nearest(padded_sector, _MODEL_INPUT_SIZE)
print(f"pad_mask     : {prev_pad_mask_shape} -> {pad_mask_r.shape}")
print(f"padded_sector: {prev_sector_shape} -> {sector_r.shape}")

### 4. Tensor conversion, ImageNet normalisation, sector masking

Final preprocessing steps:

1. **Channel expansion.** Repeat the grayscale channel three times so the shape becomes `(3, H, W)` — the ImageNet layout.
2. **ImageNet z-score normalisation.** `(x - mean) / std` per channel, using the classic ImageNet statistics.
3. **Sector masking.** Combine the pad mask and the sector mask, then use `torch.where` to force every out-of-sector pixel to exactly `0.0`.

This is where the synthetic white boxes from Section 4 disappear: they sit in the background, outside the sector, so step 3 nulls them. The middle panel below shows the normalised frame (boxes still visible); the right panel shows the same tensor after sector masking (boxes gone).

In [ ]:
def prep_for_display(t: torch.Tensor) -> np.ndarray:
    """Rearrange a ``(C, H, W)`` tensor to ``(H, W, C)`` and rescale to [0, 1].

    Matplotlib expects HWC layout in the [0, 1] range; z-score normalised
    tensors have negative values and unbounded highs, so we min-max rescale
    purely for visualisation.
    """
    arr = t.permute(1, 2, 0).numpy()
    return (arr - arr.min()) / (arr.max() - arr.min())

In [ ]:
# 1. Grayscale -> 3-channel tensor scaled to [0, 1].
tensor = torch.from_numpy(padded).unsqueeze(0).expand(3, -1, -1).contiguous()
tensor = tensor / 255.0

# 2. ImageNet z-score normalisation, per channel.
mean = torch.tensor(_IMAGENET_MEAN, dtype=tensor.dtype).view(3, 1, 1)
std = torch.tensor(_IMAGENET_STD, dtype=tensor.dtype).view(3, 1, 1)
normalized = (tensor - mean) / std

# 3. Combine masks (valid = inside original frame AND inside sector) and
#    zero everything outside — this is where the corner boxes vanish.
combined = pad_mask_r.astype(bool) & sector_r.astype(bool)
mask_t = torch.from_numpy(combined).unsqueeze(0)
zeroed = torch.where(mask_t, normalized, torch.zeros_like(normalized))

print(f"Normalised tensor range: [{normalized.min():.2f}, {normalized.max():.2f}]")
print(f"Masked tensor range    : [{zeroed.min():.2f}, {zeroed.max():.2f}]")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(combined, cmap="gray")
axes[0].set_title("Combined mask\n(pad AND sector)")
axes[1].imshow(prep_for_display(normalized))
axes[1].set_title("Normalised tensor\n(boxes still visible)")
axes[2].imshow(prep_for_display(zeroed))
axes[2].set_title("Sector-masked tensor\n(boxes suppressed)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Noise generation and image perturbation

We now define the five perturbation kernels and a small orchestration layer that applies them in a fixed **canonical order**. Order matters because perturbations do not commute — e.g. adding salt-and-pepper *after* contrast attenuation would push the impulses outside the compressed dynamic range and look completely different from adding it *before*.

### Constants, parameter tables and dataclasses

In [ ]:
# Canonical application order. Kernels are always applied in this order,
# regardless of the order in which they were sampled.
CANONICAL_ORDER: tuple[str, ...] = (
    "gaussian_blur",
    "contrast",
    "speckle",
    "gaussian_noise",
    "salt_pepper",
)
_ORDER_INDEX = {name: i for i, name in enumerate(CANONICAL_ORDER)}

# Per-severity parameter tables. `contrast` is handled separately because
# it also samples a random sign for `ln_gamma`.
_MILD = {
    # sigma_axial/lateral_px: Gaussian PSF standard deviation (pixels).
    "gaussian_blur": {"sigma_lateral_px": 0.8, "sigma_axial_px": 0.5},
    # psnr_db_target: target sector PSNR after applying the kernel.
    "speckle": {"psnr_db_target": 28.0},
    "gaussian_noise": {"psnr_db_target": 30.0},
    # fraction: share of sector pixels corrupted; ratio: salt vs. pepper.
    "salt_pepper": {"fraction": 0.005, "ratio": 0.5},
}
_MODERATE = {
    "gaussian_blur": {"sigma_lateral_px": 1.2, "sigma_axial_px": 1.0},
    "speckle": {"psnr_db_target": 23.0},
    "gaussian_noise": {"psnr_db_target": 25.0},
    "salt_pepper": {"fraction": 0.02, "ratio": 0.5},
}


@dataclass(frozen=True)
class PerturbationStep:
    """A single sampled perturbation to apply."""

    type: str
    severity: str
    params: Mapping[str, Any]


@dataclass(frozen=True)
class PerturbationResult:
    """Output of :func:`apply_perturbations` for a single frame."""

    image_uint8: np.ndarray
    applied_params: tuple[dict, ...]
    psnr_db: float
    ssim: float

### Helper utilities (quantisation, validation, metrics)

In [ ]:
def _quantize(image: np.ndarray) -> np.ndarray:
    """Clip to [0, 1] and cast to uint8 with banker-safe rounding."""
    return np.rint(np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)


def _validate(image: np.ndarray, mask: np.ndarray) -> None:
    """Enforce the ``(float32 image, bool mask, matching shapes)`` contract."""
    if image.dtype != np.float32:
        raise TypeError(f"image must be float32, got {image.dtype}")
    if image.ndim != 2:
        raise ValueError(f"image must be 2-D, got shape {image.shape}")
    if mask.dtype != np.bool_:
        raise TypeError(f"mask must be bool, got {mask.dtype}")
    if mask.shape != image.shape:
        raise ValueError(
            f"mask shape {mask.shape} does not match image shape {image.shape}"
        )


def _compose_uint8(
    stage: np.ndarray, clean: np.ndarray, mask: np.ndarray
) -> np.ndarray:
    """Quantise ``stage`` inside the sector and the clean input outside."""
    out = _quantize(clean)
    stage_u8 = _quantize(stage)
    out[mask] = stage_u8[mask]
    return out

In [ ]:
def _sector_psnr(stage: np.ndarray, clean: np.ndarray, mask: np.ndarray) -> float:
    """Peak signal-to-noise ratio computed on sector pixels only."""
    if not mask.any():
        return math.inf
    diff = stage[mask].astype(np.float64) - clean[mask].astype(np.float64)
    mse = float(np.mean(diff * diff))
    if mse == 0.0:
        return math.inf
    # Compute the decibel PSNR value, where the value in the logarithm represents
    # the ratio of the maximum possible pixel value (1.0 for normalised float)
    # to the mean squared error (MSE) between the stage and clean images
    return 10.0 * math.log10(1.0 / mse)


def _sector_ssim(stage: np.ndarray, clean: np.ndarray, mask: np.ndarray) -> float:
    """SSIM over the sector bounding box (falls back to the full frame if too small)."""
    if not mask.any():
        return 1.0

    # Tight bounding box around the sector for a meaningful SSIM comparison.
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    r0, r1 = int(np.argmax(rows)), int(mask.shape[0] - np.argmax(rows[::-1]))
    c0, c1 = int(np.argmax(cols)), int(mask.shape[1] - np.argmax(cols[::-1]))
    h, w = r1 - r0, c1 - c0

    if h < _SSIM_WIN or w < _SSIM_WIN:
        if stage.shape[0] < _SSIM_WIN or stage.shape[1] < _SSIM_WIN:
            logger.warning("SSIM: frame smaller than %d px; returning 1.0", _SSIM_WIN)
            return 1.0
        logger.warning(
            "SSIM: sector bbox %dx%d smaller than %d; using full frame", h, w, _SSIM_WIN
        )
        y_crop, x_crop = stage, clean
    else:
        y_crop = stage[r0:r1, c0:c1]
        x_crop = clean[r0:r1, c0:c1]

    ssim_value = structural_similarity(
        y_crop,
        x_crop,
        data_range=1.0,
        gaussian_weights=True,
        sigma=1.5,
        use_sample_covariance=False,
    )
    return float(np.asarray(ssim_value).item())

### Kernel definitions

In [ ]:
def gaussian_blur(
    image: np.ndarray,
    mask: np.ndarray,
    params: Mapping[str, Any],
    rng: np.random.Generator,
) -> np.ndarray:
    """Anisotropic Gaussian blur applied inside the sector only.

    Uses **normalised convolution**: both ``image * mask`` and ``mask``
    are Gaussian-filtered, and the ratio is taken so that padded/outside
    pixels do not leak dark values into the sector boundary.

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        params: ``{"sigma_lateral_px": float, "sigma_axial_px": float}``.
        rng: Unused (deterministic kernel), kept for signature parity.

    Returns:
        Blurred float32 image.
    """
    _validate(image, mask)
    sigma_lat = float(params["sigma_lateral_px"])
    sigma_ax = float(params["sigma_axial_px"])
    sigma_np = (sigma_ax, sigma_lat)  # numpy uses (rows, cols) = (axial, lateral)

    # Normalised convolution: prevents dark background from leaking in.
    mask_f = mask.astype(np.float32)
    num = gaussian_filter(image * mask_f, sigma=sigma_np).astype(np.float32)
    den = gaussian_filter(mask_f, sigma=sigma_np).astype(np.float32)
    blurred = num / np.maximum(den, np.float32(1e-8))

    out = image.copy()
    out[mask] = blurred[mask]
    return out

In [ ]:
def contrast(
    image: np.ndarray,
    mask: np.ndarray,
    params: Mapping[str, Any],
    rng: np.random.Generator,
) -> np.ndarray:
    """Gamma + linear rescale around the sector mean.

    ``y = mu + s * (x**gamma - mu)`` where ``mu`` is the mean over the
    sector. ``gamma = exp(ln_gamma)`` shifts the tone curve; ``s < 1``
    attenuates contrast.

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        params: ``{"ln_gamma": float, "s": float}``.
        rng: Unused, kept for signature parity.

    Returns:
        Contrast-adjusted float32 image.
    """
    _validate(image, mask)
    gamma = float(np.exp(float(params["ln_gamma"])))
    s = float(params["s"])

    if not mask.any():
        return image.copy()

    # Tone-curve then contrast rescale around the sector mean.
    x_pow = np.power(image, gamma, dtype=np.float32)
    mu = np.float32(x_pow[mask].mean())
    y = mu + np.float32(s) * (x_pow - mu)

    out = image.copy()
    out[mask] = y[mask]
    return out

In [ ]:
def gaussian_noise(
    image: np.ndarray,
    mask: np.ndarray,
    params: Mapping[str, Any],
    rng: np.random.Generator,
) -> np.ndarray:
    """Additive white Gaussian noise, sector-scoped.

    ``y = clip(x + n, 0, 1)``, ``n ~ N(0, sigma^2)`` with ``sigma`` set
    from the target sector PSNR.

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        params: ``{"psnr_db_target": float}``.
        rng: RNG used to sample the Gaussian noise field.

    Returns:
        float32 image with AWGN added inside the sector.
    """
    _validate(image, mask)
    psnr = float(params["psnr_db_target"])

    if not mask.any():
        return image.copy()

    sector_vals = image[mask]
    sigma = float(np.sqrt(10.0 ** (-psnr / 10.0)))
    n = rng.standard_normal(size=sector_vals.shape).astype(np.float32) * np.float32(
        sigma
    )
    y = np.clip(sector_vals + n, 0.0, 1.0).astype(np.float32)

    out = image.copy()
    out[mask] = y
    return out

In [ ]:
def speckle(
    image: np.ndarray,
    mask: np.ndarray,
    params: Mapping[str, Any],
    rng: np.random.Generator,
) -> np.ndarray:
    """Signal-dependent multiplicative-like speckle (Loupas/Coupe form).

    ``y = x + sqrt(x) * n`` with ``n ~ N(0, sigma^2)``; ``sigma`` is
    derived from the target sector PSNR ``psnr_db_target``.

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        params: ``{"psnr_db_target": float}``.
        rng: RNG used to sample the Gaussian noise field.

    Returns:
        float32 image with speckle added inside the sector.
    """
    _validate(image, mask)
    psnr = float(params["psnr_db_target"])

    if not mask.any():
        logger.warning("speckle: empty sector; no-op")
        return image.copy()

    sector_vals = image[mask]
    mean_x = float(sector_vals.mean())
    if mean_x <= 0.0:
        logger.warning("speckle: mean(x[mask]) == 0; no-op")
        return image.copy()

    # Solve for sigma so that E[MSE] = 10^(-PSNR/10).
    mse_t = 10.0 ** (-psnr / 10.0)
    sigma = float(np.sqrt(mse_t / mean_x))
    n = rng.standard_normal(size=sector_vals.shape).astype(np.float32) * np.float32(
        sigma
    )
    y = sector_vals + np.sqrt(np.clip(sector_vals, 0.0, 1.0)) * n
    y = np.clip(y, 0.0, 1.0).astype(np.float32)

    out = image.copy()
    out[mask] = y
    return out

In [ ]:
def salt_pepper(
    image: np.ndarray,
    mask: np.ndarray,
    params: Mapping[str, Any],
    rng: np.random.Generator,
) -> np.ndarray:
    """Impulse noise inside the sector.

    A ``fraction`` share of sector pixels is picked without replacement;
    a ``ratio`` share of those becomes salt (1.0), the rest pepper (0.0).

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        params: ``{"fraction": float, "ratio": float}``.
        rng: RNG used to pick the impulse positions.

    Returns:
        float32 image with salt-and-pepper impulses.
    """
    _validate(image, mask)
    fraction = float(params["fraction"])
    ratio = float(params["ratio"])

    sector_indices = np.flatnonzero(mask.ravel())
    n_sector = sector_indices.size
    if n_sector == 0:
        return image.copy()

    count = int(round(fraction * n_sector))
    if count <= 0:
        return image.copy()
    count = min(count, n_sector)

    # Sample impulse locations, then split into salt vs. pepper.
    picked = rng.choice(sector_indices, size=count, replace=False)
    salt_count = max(0, min(int(round(ratio * count)), count))
    salt_idx = picked[:salt_count]
    pepper_idx = picked[salt_count:]

    out = image.copy()
    flat = out.ravel()  # shares memory with `out`
    flat[salt_idx] = np.float32(1.0)
    flat[pepper_idx] = np.float32(0.0)
    return out

### Sampling and orchestration

In [ ]:
def sample_perturbation_steps(
    type_names: Sequence[str],
    rng: np.random.Generator,
    severity: str | None = None,
    sample_perturbations: bool = False,
) -> list[PerturbationStep]:
    """Sample one :class:`PerturbationStep` per requested kernel.

    Args:
        type_names: Kernel names to sample. Not mutated.
        rng: RNG driving all sampling decisions.
        severity: ``"mild"`` or ``"moderate"``. If ``None``, the severity
            is resampled *independently* for every step.
        sample_perturbations: If ``True``, shuffle the kernels and keep
            a random non-empty subset (1..len).

    Returns:
        List of :class:`PerturbationStep`.
    """
    # Copy so we never mutate the caller's list, even under shuffle.
    names = list(type_names)
    if sample_perturbations:
        rng.shuffle(names)
        k = int(rng.integers(1, len(names) + 1))
        names = names[:k]

    steps: list[PerturbationStep] = []
    for name in names:
        # Resample severity per step when the caller left it unspecified.
        sev = (
            severity if severity is not None else str(rng.choice(["mild", "moderate"]))
        )
        if name == "contrast":
            if sev == "mild":
                magnitude = rng.uniform(0.3, 0.4)
                s = rng.uniform(0.8, 0.9)
            else:
                magnitude = rng.uniform(0.6, 0.8)
                s = rng.uniform(0.4, 0.6)
            sign = int(rng.choice([-1, 1]))
            params: Mapping[str, Any] = {
                "ln_gamma": float(sign * magnitude),
                "s": float(s),
            }
        else:
            base = _MILD if sev == "mild" else _MODERATE
            params = dict(base[name])
        steps.append(PerturbationStep(type=name, severity=sev, params=params))
    return steps

In [ ]:
KERNELS = {
    "gaussian_blur": gaussian_blur,
    "contrast": contrast,
    "speckle": speckle,
    "gaussian_noise": gaussian_noise,
    "salt_pepper": salt_pepper,
}


def apply_perturbations(
    image: np.ndarray,
    mask: np.ndarray,
    steps: Sequence[PerturbationStep],
    rng: np.random.Generator,
) -> PerturbationResult:
    """Apply ``steps`` to ``image`` inside ``mask`` in canonical order.

    Args:
        image: 2D float32 image in [0, 1].
        mask: Sector mask (bool, same shape as ``image``).
        steps: Perturbation steps to apply. Any step whose ``type`` is
            not in :data:`CANONICAL_ORDER` raises ``ValueError``.
        rng: RNG threaded through each kernel.

    Returns:
        :class:`PerturbationResult` with the final uint8 frame plus
        sector-scoped PSNR and SSIM against the clean input.
    """
    _validate(image, mask)

    unknown = [s.type for s in steps if s.type not in _ORDER_INDEX]
    if unknown:
        raise ValueError(f"Unknown perturbation types: {unknown}")

    if not mask.any():
        logger.warning("apply_perturbations: empty sector; skipping all kernels")

    # Enforce canonical order regardless of the caller's ordering.
    ordered = sorted(steps, key=lambda s: _ORDER_INDEX[s.type])

    if not ordered:
        return PerturbationResult(
            image_uint8=_quantize(image),
            applied_params=(),
            psnr_db=math.inf,
            ssim=1.0,
        )

    # Apply the kernels sequentially on a working copy.
    stage = image.astype(np.float32, copy=True)
    for step in ordered:
        stage = KERNELS[step.type](stage, mask, step.params, rng)

    return PerturbationResult(
        image_uint8=_compose_uint8(stage, image, mask),
        applied_params=tuple(dict(s.params) for s in ordered),
        psnr_db=_sector_psnr(stage, image, mask),
        ssim=_sector_ssim(stage, image, mask),
    )

## Building the perturbation gallery

We now generate one perturbed frame per kernel at **moderate** severity, save each to `../lab-0-data/perturbed_images/` and display them side by side. `noise_removal.ipynb` will pick up these files by filename.

In [ ]:
# Reproducible RNG for this section.
rng = np.random.default_rng(SEED)

# Output folder consumed by `noise_removal.ipynb`. We clear any stale
# single-kernel PNGs so the gallery stays consistent across re-runs.
OUTPUT_FILE_PATH = Path("./lab-0-data/perturbed_images")
OUTPUT_FILE_PATH.mkdir(parents=True, exist_ok=True)
for stale in OUTPUT_FILE_PATH.glob("perturbed_*_moderate_*.png"):
    stale.unlink()
print(f"Output folder ready: {OUTPUT_FILE_PATH.resolve()}")

### Kernel recap (moderate severity)

| Kernel | Physical model | Main knob | Moderate setting |
|---|---|---|---|
| `gaussian_blur` | Anisotropic PSF (normalised convolution) | axial / lateral sigma (px) | `sigma_axial = 1.0`, `sigma_lateral = 1.2` |
| `contrast` | `y = mu + s * (x**gamma - mu)` | `s`, sign of `ln_gamma` | contrast attenuation |
| `speckle` | `y = x + sqrt(x) * n` (Loupas/Coupe) | target sector PSNR (dB) | `PSNR = 23 dB` |
| `gaussian_noise` | `y = x + n`, `n ~ N(0, sigma^2)` | target sector PSNR (dB) | `PSNR = 25 dB` |
| `salt_pepper` | random impulses (0 or 1) | fraction, salt/pepper ratio | `fraction = 0.02`, `ratio = 0.5` |

Canonical order is `gaussian_blur → contrast → speckle → gaussian_noise → salt_pepper`.

In [ ]:
# Reload the clean CAMUS frame fresh from disk: `image` in Section 4
# was overwritten with synthetic white boxes, so we cannot reuse it here.
clean = dcmread(selected_file).pixel_array.astype(np.float32)
# Kernels expect a float32 image in [0, 1].
clean = (clean - clean.min()) / (clean.max() - clean.min())

# Recompute the sector mask on the clean frame.
sector_mask = compute_sector_mask(clean)

# One perturbed frame per kernel, moderate severity.
gallery: dict[str, dict] = {}
for kernel_name in CANONICAL_ORDER:
    steps = sample_perturbation_steps([kernel_name], rng, severity="moderate")
    result = apply_perturbations(clean, sector_mask, steps, rng)

    out_file = (
        OUTPUT_FILE_PATH / f"perturbed_{selected_file.stem}_moderate_{kernel_name}.png"
    )
    plt.imsave(out_file, result.image_uint8, cmap="gray", vmin=0, vmax=255)

    gallery[kernel_name] = {
        "image": result.image_uint8,
        "psnr_db": result.psnr_db,
        "ssim": result.ssim,
        "params": result.applied_params[0],
    }
    print(
        f"{kernel_name:>16s}: PSNR = {result.psnr_db:6.2f} dB, "
        f"SSIM = {result.ssim:.3f}  ->  {out_file.name}"
    )

### Gallery — one kernel per panel

Each panel title reports the sector-scoped PSNR (dB) and SSIM against the clean input — a quick numeric sanity check that all five perturbations sit in a comparable severity band.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
axes = axes.ravel()

# Panel 0: clean reference (float32 in [0, 1]).
axes[0].imshow(clean, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Clean (float32, normalised)", fontsize=10)
axes[0].axis("off")

# Panels 1..5: one perturbed frame per kernel, in canonical order.
for ax, kernel_name in zip(axes[1:], CANONICAL_ORDER):
    info = gallery[kernel_name]
    ax.imshow(info["image"], cmap="gray", vmin=0, vmax=255)
    ax.set_title(
        f"{kernel_name}\nPSNR = {info['psnr_db']:.1f} dB, SSIM = {info['ssim']:.2f}",
        fontsize=10,
    )
    ax.axis("off")

fig.suptitle(
    "Perturbation gallery — one kernel per panel (moderate severity)",
    fontsize=12,
    fontweight="bold",
)
plt.show()

### Severity dial — mild vs. moderate

Each kernel exposes a `mild` / `moderate` severity that maps to the `_MILD` / `_MODERATE` parameter tables above. Here we compare the two severities on a single kernel — `gaussian_noise` — where the 5 dB PSNR drop between mild (~30 dB) and moderate (~25 dB) roughly doubles the perceived noise floor.

In [ ]:
demo_kernel = "gaussian_noise"
severities = ("mild", "moderate")
# Fresh RNG so this comparison is reproducible independently of the gallery loop.
sev_rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
axes[0].imshow(clean, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Clean", fontsize=10)
axes[0].axis("off")

for ax, sev in zip(axes[1:], severities):
    steps = sample_perturbation_steps([demo_kernel], sev_rng, severity=sev)
    res = apply_perturbations(clean, sector_mask, steps, sev_rng)
    ax.imshow(res.image_uint8, cmap="gray", vmin=0, vmax=255)
    ax.set_title(
        f"{demo_kernel} - {sev}\nPSNR = {res.psnr_db:.1f} dB, SSIM = {res.ssim:.2f}",
        fontsize=10,
    )
    ax.axis("off")

fig.suptitle(
    f"Severity dial for '{demo_kernel}' - mild vs. moderate",
    fontsize=12,
    fontweight="bold",
)
plt.show()

---

In `noise_removal.ipynb` you will pick each of these five files from `../lab-0-data/perturbed_images/`, identify the underlying noise type from a homogeneous ROI histogram and apply a targeted counter-filter.